# Copyright & Intellectual Property— Copyright as Engineering Checks

**Module 15 · Lesson 15.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Training-Data Question
- Memorization & Regurgitation
- Fair Use: The Four Factors
- Substantial Similarity
- Licence Compatibility
- Provenance & the TDM Opt-Out

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

## Copyright is not a lawyer’s problem you inherit at launch

> 💡 **Info**
>
> The most expensive copyright mistakes in GenAI are not subtle points of law — they are missing engineering checks. A model that quotes a source article **verbatim** into a user’s face; a corpus nobody can account for when a rightsholder asks “was my work in there?”; a **GPL** dependency that quietly obligates you to open-source your whole product; a provider indemnity you assumed was a blanket shield but was conditional on guardrails you skipped. Each of these is discoverable *before* launch with a small amount of code and discipline, and each is ruinous to discover *after*, in a letter from a lawyer. This lesson treats copyright the way you already treat reliability and security: as a set of checks you build into the release. You will quantify how much of your training data is copyrighted, run an output-side memorization filter, score the four fair-use factors, measure substantial similarity against the de-minimis line courts tolerate, verify licence compatibility across your stack, keep a per-document provenance ledger so you can honour a text-and-data-mining opt-out, and finally gate the release on the conditions that keep your provider indemnity valid. None of it makes you a lawyer — it makes you the engineer who does not hand the lawyer a disaster.

### 🎯 What you will be able to do after this lesson

- **Explain** why training on copyrighted data is contested, and how memorization turns a grey-area input into a clear-infringement output.

- **Apply** the four-factor fair-use test and a substantial-similarity measure to judge whether a use or an output carries infringement risk.

- **Analyze** a licence stack for compatibility and a provenance ledger for the opt-outs you must exclude.

- **Evaluate** a release against a compliance gate, and judge when provider indemnity actually applies and when an output has no copyright at all.

> 📦 **Info**
>
> ✅ Before you startThis lesson follows **15.2 (Explainable AI)**: there you learned to explain what a model *used* to decide. Copyright is a different axis of accountability — it is about the *rights* in the data that went in and the outputs that come out, not the model’s reasoning. It also builds on **Module 08** (RAG grounds answers in retrieved source documents — and quoting a retrieved passage back is a memorization-style risk) and **Module 03** (a model is trained on a huge scraped corpus, mostly copyrighted, mostly unlicensed). Everything here is keyless arithmetic on illustrative facts, and it is engineering education, not legal advice.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🎵 **Analogy**
>
> Training on copyrighted data is like **sampling in music**. A producer can lift a two-second drum break from an old record and build a whole new track on it — and whether that is a brilliant transformation or theft has been fought in court for decades. Sometimes it is cleared and licensed; sometimes it is a lawsuit. A model trained on scraped text is sampling at planetary scale: it ingests millions of protected works and produces something new from them. The unresolved question is the same one the music industry spent thirty years on — when does building on someone else’s protected material become infringement? **Where it breaks down:** a sample is one identifiable source you can point to and clear; a model blends millions, which is what makes both the licensing and the memorization questions so much harder.

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“If we did not copy it word-for-word, we are safe.”** Substantial similarity catches a paraphrase that keeps the protected *expression*. Verbatim copying is only the easy case.
> - **“The AI made it, so we own the copyright.”** A purely AI-generated output has no human author and therefore no copyright in the US (US Copyright Office; Thaler v Perlmutter) — you cannot stop others from copying it.
> - **“Training on public web data is automatically fair use.”** It is contested, not settled — and recent rulings weigh market harm and how the corpus was acquired heavily against a blanket fair-use claim.

> 💡 **Info**
>
> ⚠️ Anti-patternShipping with no provenance ledger and no output-side memorization filter, then assuming the cloud provider’s copyright indemnity will cover any claim. A team scrapes broadly, keeps no record of what went in, never checks outputs for reproduced passages, and treats the provider’s “copyright commitment” as insurance — only to find, when the letter arrives, that the indemnity was conditional on exactly the content filters and guardrails they turned off. The result is the worst case: no way to honour an opt-out, no defence against a verbatim-quote claim, and no coverage. Every step below is one of the checks that turns this disaster into a launch gate.

---

## 🎯 Concept 1: The Training-Data Question

### The Training-Data Question

Models learn from scraped text, most of it copyrighted, and whether that TRAINING is lawful is the central, unsettled fight. Quantify the exposure first: how much of your corpus is copyright-encumbered? Developers argue transformative fair use; rightsholders argue mass infringement plus market harm - and courts are split.

Start where the risk starts: the training data. A frontier model learns from an enormous corpus scraped from the open web, and most of that text is **copyrighted** — news articles, books, code, art, forum posts — used without an individual licence. Whether training on it is lawful is *the* unresolved question of GenAI, and it is genuinely contested. Developers argue that training is **transformative fair use**: the model learns statistical patterns, it does not store and resell the works. Rightsholders argue it is **mass infringement** plus **market harm**: their work was copied wholesale to build a product that competes with them. The courts are split — NYT v OpenAI is pending, Getty v Stability is live, and the 2025 book cases split on how the corpus was acquired. Because you cannot resolve this in code, you do the engineering thing: *quantify the exposure* so you know how much of your corpus is copyright-encumbered and can decide how much risk you are carrying. In the worked example, of 1,000 documents, 720 are copyrighted-and-unlicensed, 180 are licensed, and 100 are public domain — so 72 percent of the corpus is exposed. That number does not settle the law, but it tells you the size of the bet, and it is the input to every mitigation that follows. The block computes the exposure, keyless.

> 🌐 **Analogy**
>
> The training-data question is **building a house on land with an unclear title**. You can pour the foundation and frame the walls, and it may all be fine — but if the title was never cleared, someone can show up later with a deed and a claim on what you built. A prudent builder does a title search *first*: how much of this land is actually clear, how much is contested, how much is provably yours? Quantifying your copyrighted-training exposure is that title search. It does not make the contested land yours, but it tells you exactly how much of your foundation sits on ground someone else might claim — which is the difference between a calculated risk and a blind one.

Of 1,000 training documents, 720 are copyrighted-and-unlicensed, 180 licensed, 100 public domain. What does the 72 percent exposure figure tell you?

**📝 `01_training_data_exposure.py`** - *runnable*

In [ ]:
# THE TRAINING-DATA QUESTION: models learn from scraped text, most of it copyrighted. Whether that TRAINING is lawful is the
# central fight (NYT v OpenAI, Getty v Stability). Quantify the exposure first: how much of your corpus is copyright-encumbered?
corpus = {"copyrighted_scraped": 720, "licensed": 180, "public_domain": 100}   # documents, illustrative
total = sum(corpus.values())
print("Training corpus of {:,} documents:".format(total))
for k, v in corpus.items():
    print("  {:<22} {:>4}  ({:.0%})".format(k, v, v / total))
exposed = corpus["copyrighted_scraped"] / total
print()
print("{:.0%} of the corpus is copyrighted material used WITHOUT a licence. Two legal positions collide:".format(exposed))
print("  - developers argue TRAINING is transformative fair use (learning patterns, not copying expression);")
print("  - rightsholders argue it is mass infringement plus market harm. Courts are split; treat unlicensed scraping as a RISK,")
print("not a settled right. The mitigations in this lesson (memorization checks, licensing, provenance, indemnity) manage that risk.")

# Output:
# Training corpus of 1,000 documents:
#   copyrighted_scraped     720  (72%)
#   licensed                180  (18%)
#   public_domain           100  (10%)
#
# 72% of the corpus is copyrighted material used WITHOUT a licence. Two legal positions collide:
#   - developers argue TRAINING is transformative fair use (learning patterns, not copying expression);
#   - rightsholders argue it is mass infringement plus market harm. Courts are split; treat unlicensed scraping as a RISK,
# not a settled right. The mitigations in this lesson (memorization checks, licensing, provenance, indemnity) manage that risk.

- The corpus splits into copyrighted-scraped (720), licensed (180), and public-domain (100) documents.
- So **72 percent** of the corpus is copyrighted material used without a licence — the exposure.
- Two positions collide: developers argue transformative fair use; rightsholders argue mass infringement and market harm.
- The number does not settle the law (courts are split) — it sizes the risk and feeds every mitigation that follows.

#### 💡 What just happened

⚡ What just happened? Training on copyrighted data is the central unsettled fight; you cannot resolve it in code, so you quantify the exposure (here 72 percent) to size the risk. Tradeoff: none - measuring the exposure is the first control. Next: the input question is contested, but a verbatim output is a clearer problem.

- A 1000-doc corpus splits into copyrighted / licensed / public-domain
- A percent-exposed readout shows 72 percent copyrighted without a licence

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Memorization & Regurgitation

### Memorization & Regurgitation

Training on a work is arguable; emitting it VERBATIM is clearer infringement. Detect it with the longest run of shared words between an output and a source - a plagiarism check for outputs. Extraction attacks show large models can be induced to reproduce training text, so an output-side filter is a real defence.

The training question is a grey area; the **output** question can be black and white. If your model emits a passage from a copyrighted work *verbatim*, that is a reproduction, and it is a far clearer infringement than the training that produced it. This is not hypothetical: **extraction attacks** (Carlini et al.) showed that large models can be induced to regurgitate memorized training text, including copyrighted passages and even personal data. So you add an output-side check — a plagiarism detector for your own model. The simplest reliable signal is the **longest run of consecutive shared words** between an output and a candidate source: a short shared run happens by chance, but a long one does not. In the worked example the longest verbatim run between the output and the source is 11 words — well over a threshold of 6 — so the output is **flagged as memorization** and blocked or filtered before it reaches the user. This is exactly the defence against the “the model quoted our article” claim: even if you cannot prove your training was clean, you can prove your *outputs* do not reproduce protected passages. The block implements the longest-shared-run detector with no library, keyless.

> 🔍 **Analogy**
>
> A memorization filter is a **plagiarism checker run on your own model’s homework**. A university does not trust that every essay is original; it runs each one through a detector that finds long passages matching a known source, because a sentence or two overlapping is coincidence but a whole paragraph is copying. You point the same detector at your model’s outputs before they ship: a short shared phrase is fine, but a long verbatim run against a copyrighted source is the model handing in someone else’s paragraph. Catch it at the output, and the copied passage never reaches the user — the same way a plagiarism check catches the essay before it is graded.

An output shares an 11-word verbatim run with a source; the threshold is 6 words. What do you conclude and do?

**📝 `02_memorization_detector.py`** - *runnable*

In [ ]:
# MEMORIZATION / REGURGITATION: training on a work is contested; reproducing it VERBATIM is clearer infringement. Detect it by
# the longest run of shared words between an output and a source - a plagiarism check for model outputs. No library needed.
source = "It was the best of times it was the worst of times it was the age of wisdom".split()
output = "In summary it was the best of times it was the worst of times as Dickens wrote".split()
def longest_common_run(a, b):
    best = 0
    for i in range(len(a)):
        for j in range(len(b)):
            k = 0
            while i + k < len(a) and j + k < len(b) and a[i + k] == b[j + k]:
                k += 1
            best = max(best, k)
    return best
run = longest_common_run(source, output)
THRESHOLD = 6                                       # a run this long is very unlikely by chance -> memorization
print("longest verbatim shared run: {} words.".format(run))
print("  '{}'".format(" ".join(output[output.index("it"):output.index("it") + run]) if "it" in output else ""))
print("threshold = {} words -> {}".format(THRESHOLD, "MEMORIZATION flagged (likely reproduces a protected expression)" if run >= THRESHOLD else "no memorization"))
print()
print("Training on data is arguable; emitting a long verbatim copy is not. Run this check on outputs and block or filter the ones")
print("that reproduce a protected passage - the same defence that keeps you out of the NYT-style 'the model quoted our article' claim.")

# Output:
# longest verbatim shared run: 11 words.
#   'it was the best of times it was the worst of'
# threshold = 6 words -> MEMORIZATION flagged (likely reproduces a protected expression)
#
# Training on data is arguable; emitting a long verbatim copy is not. Run this check on outputs and block or filter the ones
# that reproduce a protected passage - the same defence that keeps you out of the NYT-style 'the model quoted our article' claim.

- The detector finds the **longest run of consecutive shared words** between the output and the source.
- Here the longest verbatim run is **11 words** — a short run happens by chance, a long one does not.
- The threshold is 6 words; 11 exceeds it, so the output is **flagged as memorization** and blocked or filtered.
- This is the defence against “the model quoted our article”: even with contested training, your outputs do not reproduce protected passages.

#### 💡 What just happened

⚡ What just happened? Training on a work is arguable, but emitting it verbatim is clearer infringement; a longest-shared-run detector flags a memorized passage (11 words over a 6-word threshold) so you filter it. Tradeoff: the filter needs candidate sources to compare against and adds output latency. Next: for the grey areas, the legal test that decides fair use.

- A source and an output shown as word rows
- The longest shared verbatim run (11 words) is highlighted against a threshold

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Fair Use: The Four Factors

### Fair Use: The Four Factors

US courts weigh four fair-use factors (17 U.S.C. 107): purpose, nature, amount, and market effect. Score each and read the lean - it is a BALANCING test, not a checklist. In recent AI rulings the 4th factor (market harm) carries the most weight; “transformative” helps but does not automatically win.

When a use is neither clearly infringing nor clearly licensed, US law resolves it with the **fair-use** doctrine, and fair use has a structure: **17 U.S.C. 107** lists four factors a court weighs. First, the **purpose** of the use — is it transformative, is it commercial? Second, the **nature** of the original — factual and published works get less protection than creative unpublished ones. Third, the **amount** used — how much, and did you take the “heart” of the work? Fourth, the **market effect** — does your use substitute for the original and harm its market? The crucial thing for an engineer to understand is that this is a *balancing* test, not a checklist: you do not win by counting factors in your favour. A useful mental model is to score each factor from strongly-against to strongly-for and read the lean. In the worked example, a transformative research tool scores well across the board and comes out clearly fair; a commercial product that reproduces enough to substitute for the original scores badly on purpose and market effect and comes out clearly infringing. And the fourth factor — market harm — carries the most weight in recent AI rulings (Thomson Reuters v Ross turned largely on it): being “transformative” helps the first factor but does not automatically win the case. The block scores both uses and reads the balance, keyless.

> ⚖️ **Analogy**
>
> Fair use is a **balance scale with four weights, not a four-item checklist**. You do not get a pass by putting three of the four weights on your side of the pan and declaring victory — the scale weighs the *total*, and one heavy weight can tip it by itself. Market harm is the heaviest weight on the scale: pile everything else in your favour, and if your use clearly substitutes for the original and eats its market, the scale still tips against you. This is why “but it is transformative!” is not a magic word. Transformative use is a real weight on your side of the pan, but the judge reads the whole balance, and in recent AI cases the market-harm weight has been the one that decides which way it falls.

A commercial product reproduces enough of a work to substitute for buying it, though its use is somewhat transformative. How does the four-factor test resolve?

**📝 `03_fair_use_four_factors.py`** - *runnable*

In [ ]:
# THE FAIR-USE FOUR FACTORS (17 U.S.C. 107): US courts weigh four factors. Score each from -2 (against fair use) to +2 (for it),
# sum, and read the lean. It is a BALANCING test, not a checklist - one strong factor (market harm) can sink an otherwise-fair use.
FACTORS = ["purpose (transformative? non-commercial?)", "nature (factual? published?)",
           "amount (how much, and the heart of the work?)", "market effect (a substitute?)"]
def fair_use(scores):
    total = sum(scores)
    lean = "likely FAIR USE" if total >= 2 else ("likely INFRINGING" if total <= -2 else "genuinely uncertain")
    return total, lean
research = [2, 1, 1, 2]      # a transformative research tool that does not substitute for the original
substitute = [-1, -1, 0, -2] # a commercial product that reproduces enough to replace buying the original
for name, s in [("research/transformative use", research), ("commercial substitute", substitute)]:
    total, lean = fair_use(s)
    print("{}:".format(name))
    for f, v in zip(FACTORS, s):
        print("  {:+d}  {}".format(v, f))
    print("  -> total {:+d}  ->  {}\n".format(total, lean))
print("The 4th factor (market harm) carries the most weight in recent AI rulings. 'Transformative' helps but does not automatically win.")

# Output:
# research/transformative use:
#   +2  purpose (transformative? non-commercial?)
#   +1  nature (factual? published?)
#   +1  amount (how much, and the heart of the work?)
#   +2  market effect (a substitute?)
#   -> total +6  ->  likely FAIR USE
#
# commercial substitute:
#   -1  purpose (transformative? non-commercial?)
#   -1  nature (factual? published?)
#   +0  amount (how much, and the heart of the work?)
#   -2  market effect (a substitute?)
#   -> total -4  ->  likely INFRINGING
#
# The 4th factor (market harm) carries the most weight in recent AI rulings. 'Transformative' helps but does not automatically win.

- The four factors (17 U.S.C. 107) are scored from against (-2) to for (+2) and summed to read the lean.
- A transformative research tool scores +2/+1/+1/+2 = +6 → likely **fair use**.
- A commercial substitute scores -1/-1/0/-2 = -4 → likely **infringing**.
- It is a balancing test: the 4th factor (market harm) weighs most, so “transformative” helps but does not automatically win.

#### 💡 What just happened

⚡ What just happened? Fair use weighs four factors (17 U.S.C. 107) as a balancing test: research/transformative use leans fair (+6), a commercial substitute leans infringing (-4), and market harm is the heaviest factor. Tradeoff: the scores are a judgement call, not a formula - only a court decides. Next: even a non-verbatim, non-substituting output can still infringe if it is too similar.

- A balance scale with the four factors as weights
- The scale tips toward fair use or infringing as the scores change

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Substantial Similarity

### Substantial Similarity

A NON-verbatim output can still infringe if it is substantially similar to a protected work. The idea-expression line governs: ideas are free, expression is protected. Measure the overlap of distinctive content against the de-minimis level courts tolerate - a similar plot is fine, reusing the distinctive wording is the risk.

Memorization (Step 2) catches the easy case: an exact copy. But copyright infringement does not require word-for-word copying — an output can be **substantially similar** to a protected work while changing the words, and still infringe. The governing principle is the **idea-expression distinction**: copyright protects the specific *expression* of a work, not the underlying *ideas*. A story about a young wizard at a magic school is an idea, free for anyone; the distinctive characters, plot beats, and wording J.K. Rowling used are protected expression. So a close paraphrase that keeps the protected expression — the same distinctive details, reworded — can cross the line, while a work that shares only the general idea does not. You approximate this by measuring the overlap of *distinctive* content (here a Jaccard overlap of meaningful tokens) and comparing it to the **de-minimis** level courts tolerate as trivial. In the worked example, a close paraphrase of a protected work scores 0.316 — above the substantial-similarity line of 0.30, so it is flagged as an infringement risk — while an unrelated work scores 0.000 and is clearly de minimis. The measure is a heuristic, not a verdict (a court decides), but it flags the outputs that keep too much of someone’s protected expression. The block computes the similarity, keyless.

> 🎤 **Analogy**
>
> Substantial similarity is the line between a **cover song and a copy of the recording**. You are free to write a song about heartbreak in a small town — that is an idea, and no one owns it. You can even perform your own version of someone’s melody if you license it. But if your “new” song reproduces the distinctive hook, the specific lyrics, and the arrangement of an existing hit, changing a word here and there, you have copied the protected expression, not just borrowed the idea — and no amount of “but the words are different” saves you. Substantial similarity measures how much of the distinctive expression survived the paraphrase. A shared theme is a cover; a reworded hook is a copy.

A close paraphrase scores 0.316 substantial similarity against a protected work (de-minimis 0.20, substantial line 0.30). It copies nothing word-for-word. Is it safe?

**📝 `04_substantial_similarity.py`** - *runnable*

In [ ]:
# SUBSTANTIAL SIMILARITY: even a NON-verbatim output can infringe if it is substantially similar to a protected work. Measure the
# overlap of distinctive content (Jaccard over meaningful tokens) and compare to the de minimis line courts tolerate.
def tokens(s):
    stop = {"the", "a", "an", "of", "and", "to", "in", "is", "it", "was", "as"}
    return set(w.lower().strip(".,") for w in s.split() if w.lower() not in stop)
original = "A young wizard with a lightning scar attends a secret school of magic and fights a dark lord"
para_a = "A teenage sorcerer bearing a lightning scar enrolls at a hidden magic school to battle a dark lord"  # close paraphrase
para_b = "A detective in a rainy city solves a murder using forensic science and sharp deduction"               # unrelated
def jaccard(x, y):
    a, b = tokens(x), tokens(y)
    return round(len(a & b) / len(a | b), 3)
DE_MINIMIS = 0.20
for name, text in [("close paraphrase", para_a), ("unrelated work", para_b)]:
    sim = jaccard(original, text)
    verdict = "SUBSTANTIALLY SIMILAR (infringement risk)" if sim >= 0.30 else ("de minimis / OK" if sim < DE_MINIMIS else "gray zone - review")
    print("{:<18} similarity {:.3f}  ->  {}".format(name, sim, verdict))
print()
print("Verbatim copying is the easy case; substantial similarity catches the paraphrase that keeps the protected EXPRESSION. Ideas")
print("are free, expression is protected - so a similar plot is fine, but reusing the distinctive wording and details is the risk.")

# Output:
# close paraphrase   similarity 0.316  ->  SUBSTANTIALLY SIMILAR (infringement risk)
# unrelated work     similarity 0.000  ->  de minimis / OK
#
# Verbatim copying is the easy case; substantial similarity catches the paraphrase that keeps the protected EXPRESSION. Ideas
# are free, expression is protected - so a similar plot is fine, but reusing the distinctive wording and details is the risk.

- Substantial similarity measures the overlap of **distinctive content** (a Jaccard overlap of meaningful tokens).
- A close paraphrase scores **0.316** — above the substantial line (0.30), so it is flagged as an infringement risk.
- An unrelated work scores **0.000** — below the de-minimis level (0.20), clearly fine.
- Ideas are free, expression is protected: a similar plot is fine, but reusing the distinctive wording and details is the risk.

#### 💡 What just happened

⚡ What just happened? A non-verbatim output can still infringe if it is substantially similar; measuring distinctive-content overlap (paraphrase 0.316 vs unrelated 0.000) flags the risk, on the idea-expression line. Tradeoff: the similarity score is a heuristic flag, not a legal verdict. Next: your own model is only one component - the licences of everything you assemble must also combine.

- A similarity meter with a de-minimis line and a substantial-similarity line
- A close paraphrase (0.316) crosses the line; an unrelated work (0.000) does not

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Licence Compatibility

### Licence Compatibility

Your model, its training data, and its dependencies each carry a licence, and they do not all combine. Permissive (MIT/Apache/CC-BY) combines freely; copyleft (GPL) INFECTS a derivative work and can force your whole product open; non-commercial (CC-BY-NC) forbids selling. One bad dependency is a supply-chain bug that can sink a launch.

A GenAI product is an assembly: a base model, fine-tuning data, evaluation sets, libraries, and more — and *every one of them carries a licence*. The mistake is to treat licences as boilerplate; they are executable constraints that determine whether you can legally ship, and they do not all combine. The categories that matter: **permissive** licences (MIT, Apache-2.0, CC-BY) let you build almost anything, including closed commercial products; **copyleft** licences (the GPL family) “infect” a derivative work — if you build on GPL code and distribute the result, you must release your whole product under the GPL too; **non-commercial** licences (CC-BY-NC) forbid selling the result at all; and **proprietary** data simply cannot be redistributed. In the worked example you want to ship a closed-source commercial product built from an Apache-2.0 model (fine), CC-BY data (fine), a GPL-3.0 helper library (a problem), and a CC-BY-NC eval set (a problem) — and the checker returns **BLOCKED with two conflicts**: the GPL library would force your product open-source, and the non-commercial eval set forbids the commercial use. Either conflict alone sinks the launch. This is a supply-chain check, exactly like scanning dependencies for vulnerabilities: one incompatible licence deep in the stack can force-open or block your product, and you want to catch it in CI, not in a lawsuit. The block runs the compatibility check, keyless.

> 🧱 **Analogy**
>
> Licences are **Lego bricks that do or do not snap together**. Most bricks click cleanly — permissive licences combine with almost anything. But some bricks are a different system: a copyleft (GPL) brick is like a piece that, once you attach it, forces every brick touching it to become the same kind — snap it into your closed product and the whole model has to “go GPL.” A non-commercial brick physically will not fit into anything you intend to sell. You cannot tell by glancing at the pile; you have to check each brick’s system before you build, because discovering mid-assembly that one brick forces you to rebuild the entire thing — or that you cannot sell what you made — is exactly the supply-chain surprise a licence check prevents.

You want to ship a closed-source commercial product from an Apache-2.0 model, CC-BY data, a GPL-3.0 library, and a CC-BY-NC eval set. Can you ship as-is?

**📝 `05_license_compatibility.py`** - *runnable*

In [ ]:
# LICENSE COMPATIBILITY: your model, its training data, and its dependencies each carry a licence, and they do not all combine.
# Copyleft (GPL) "infects" a derivative work; permissive (MIT/Apache) does not. Check before you ship, not after.
# category: permissive (combine freely), copyleft (forces the whole work open), proprietary (cannot redistribute), noncommercial
LICENSE = {"MIT": "permissive", "Apache-2.0": "permissive", "GPL-3.0": "copyleft",
           "CC-BY": "permissive", "CC-BY-NC": "noncommercial", "proprietary": "proprietary"}
def can_ship_commercial_closed(components):
    problems = []
    for name, lic in components:
        cat = LICENSE[lic]
        if cat == "copyleft": problems.append("{} ({}) forces your whole product open-source".format(name, lic))
        if cat == "noncommercial": problems.append("{} ({}) forbids commercial use".format(name, lic))
        if cat == "proprietary": problems.append("{} ({}) cannot be redistributed".format(name, lic))
    return problems
stack = [("base model", "Apache-2.0"), ("fine-tune data", "CC-BY"), ("a helper lib", "GPL-3.0"), ("eval set", "CC-BY-NC")]
print("shipping a closed-source commercial product built from:")
for name, lic in stack:
    print("  {:<16} {:<12} ({})".format(name, lic, LICENSE[lic]))
problems = can_ship_commercial_closed(stack)
print()
print("verdict: {}".format("CLEAR to ship" if not problems else "BLOCKED - {} conflict(s):".format(len(problems))))
for p in problems:
    print("   - " + p)
print("License incompatibility is a supply-chain bug: one GPL dependency or one non-commercial dataset can force-open or sink a product.")

# Output:
# shipping a closed-source commercial product built from:
#   base model       Apache-2.0   (permissive)
#   fine-tune data   CC-BY        (permissive)
#   a helper lib     GPL-3.0      (copyleft)
#   eval set         CC-BY-NC     (noncommercial)
#
# verdict: BLOCKED - 2 conflict(s):
#    - a helper lib (GPL-3.0) forces your whole product open-source
#    - eval set (CC-BY-NC) forbids commercial use
# License incompatibility is a supply-chain bug: one GPL dependency or one non-commercial dataset can force-open or sink a product.

- Each component carries a licence category: permissive (Apache, CC-BY), copyleft (GPL-3.0), non-commercial (CC-BY-NC).
- For a closed-source commercial product the checker returns **BLOCKED — 2 conflicts**.
- GPL-3.0 (copyleft) would force your whole product open-source; CC-BY-NC forbids the commercial use.
- Licence incompatibility is a supply-chain bug: one bad dependency deep in the stack can force-open or sink a product — catch it in CI.

#### 💡 What just happened

⚡ What just happened? Every component carries a licence, and they do not all combine: copyleft (GPL) infects a derivative and non-commercial (CC-BY-NC) forbids selling, so the stack is BLOCKED with 2 conflicts. Tradeoff: strict licence hygiene rules out some convenient dependencies and datasets. Next: beyond licences, rightsholders can now demand you exclude their work entirely.

- Four licence blocks for a closed commercial product
- GPL copyleft infects and CC-BY-NC blocks — the stack is BLOCKED

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Provenance & the TDM Opt-Out

### Provenance & the TDM Opt-Out

2026 rules (EU AI Act Art. 53, DSM Directive Art. 4) let rightsholders reserve their work from text-and-data mining - an opt-out. If a source opted out, you must EXCLUDE it. You cannot honour an opt-out or answer “what were you trained on?” after the fact without a per-document provenance ledger recorded at ingestion.

The 2026 regulatory landscape added a concrete, machine-level obligation on top of the fair-use debate. Under the EU’s **DSM Directive Article 4**, rightsholders can **reserve** their works from text-and-data mining — a machine-readable **opt-out** (in robots.txt, an HTTP header, or metadata). If a source opted out and you scraped it anyway, you are on the wrong side of the rule, and you must **exclude** it from training. Layered on top, the **EU AI Act Article 53** requires general-purpose model providers to publish a sufficiently detailed *training-data summary* and adopt a policy to respect EU copyright. Both obligations share a prerequisite that is pure engineering: **provenance**. You cannot honour an opt-out you have no record of, and you cannot summarise training data you never tracked. In the worked example a five-source **provenance ledger** records each source, its licence, and whether it opted out: two sources (a news site and an artist portfolio) reserved their rights and must be excluded, one has an unknown licence to verify, and two are cleared. The point is that this ledger has to be built *at ingestion*, per document — trying to reconstruct “what were we trained on?” after the fact, across a scraped corpus of billions of documents, is effectively impossible. Provenance is the foundation the whole compliance story stands on. The block builds the ledger and flags the exclusions, keyless.

> 🧾 **Analogy**
>
> A provenance ledger is a **chain-of-custody receipt for every document**. In a courtroom, evidence is only admissible if every hand it passed through is logged — where it came from, who handled it, when. Skip the chain of custody and the evidence is worthless, no matter how real it is, because you cannot prove its history. A training corpus needs the same discipline: for every document, where did it come from, what is its licence, did the owner opt out? Build that receipt at the moment you ingest each document and you can answer any later question — honour an opt-out, produce a training-data summary, prove a source was cleared. Try to reconstruct the chain of custody afterward, across billions of documents, and you have nothing.

A rightsholder used the DSM Art. 4 machine-readable opt-out to reserve their site from mining, and it is in your corpus. What must you do, and what makes it possible?

**📝 `06_provenance_opt_out.py`** - *runnable*

In [ ]:
# PROVENANCE + THE TDM OPT-OUT: 2026 rules (EU AI Act Art. 53 + DSM Directive Art. 4) let rightsholders RESERVE their work from
# text-and-data mining (an opt-out, e.g. in robots.txt or a header). If a source opted out, you must EXCLUDE it - so track provenance.
sources = [   # (source, licence, opted_out_of_TDM)
    ("openwebtext dump", "unknown", False),
    ("nytimes.com articles", "copyrighted", True),     # reserved rights -> must exclude
    ("wikipedia", "CC-BY-SA", False),
    ("licensed news partner", "commercial licence", False),
    ("artiststation portfolios", "copyrighted", True)]  # opted out
print("data provenance ledger (source, licence, TDM opt-out):")
must_exclude, unknown = [], []
for src, lic, opt in sources:
    flag = "EXCLUDE (rights reserved)" if opt else ("verify licence" if lic == "unknown" else "OK to use")
    print("  {:<26} {:<20} {}".format(src, lic, flag))
    if opt: must_exclude.append(src)
    if lic == "unknown": unknown.append(src)
print()
print("{} source(s) reserved rights (opt-out) and must be removed from training; {} have unknown licences to verify.".format(len(must_exclude), len(unknown)))
print("You cannot honour an opt-out or answer 'what were you trained on?' (EU AI Act Art. 53 training-data summary) without a")
print("provenance ledger. Provenance is the foundation - keep it per document, at ingestion, or you cannot comply after the fact.")

# Output:
# data provenance ledger (source, licence, TDM opt-out):
#   openwebtext dump           unknown              verify licence
#   nytimes.com articles       copyrighted          EXCLUDE (rights reserved)
#   wikipedia                  CC-BY-SA             OK to use
#   licensed news partner      commercial licence   OK to use
#   artiststation portfolios   copyrighted          EXCLUDE (rights reserved)
#
# 2 source(s) reserved rights (opt-out) and must be removed from training; 1 have unknown licences to verify.
# You cannot honour an opt-out or answer 'what were you trained on?' (EU AI Act Art. 53 training-data summary) without a
# provenance ledger. Provenance is the foundation - keep it per document, at ingestion, or you cannot comply after the fact.

- The provenance ledger records each source, its licence, and whether it opted out of text-and-data mining.
- Two sources reserved their rights (a news site and an artist portfolio) and must be **excluded** from training.
- One source has an unknown licence to verify; two are cleared to use.
- You cannot honour an opt-out or produce the EU AI Act Art. 53 training-data summary without this ledger — built per document, at ingestion.

#### 💡 What just happened

⚡ What just happened?2026 rules let rightsholders opt out of text-and-data mining (DSM Art. 4); you must exclude opted-out sources, and only a per-document provenance ledger lets you honour that and answer the EU AI Act Art. 53 training-data summary. Tradeoff: provenance tracking is real engineering overhead at ingestion time - but it is impossible to add after the fact. Next: tie every check together into one release gate.

- A five-row provenance ledger (source, licence, opt-out)
- Two opted-out rows are flagged EXCLUDE, one unknown to verify

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Compliance Gate & Indemnity

### The Compliance Gate & Indemnity

Providers now offer copyright INDEMNITY (Microsoft Copilot Copyright Commitment, OpenAI Copyright Shield) - but only if you use the guardrails. Gate the release on provenance, opt-out exclusion, a memorization filter, licence-conflict resolution, and human authorship. And a pure-AI output is not copyrightable (US Copyright Office), so you cannot stop others copying it.

The final step turns the previous six into one decision: does this release ship? Major providers now offer **copyright indemnity** — the Microsoft Copilot Copyright Commitment, Google’s and Amazon’s indemnities, the OpenAI Copyright Shield — promising to defend enterprise customers against copyright claims arising from the provider’s model. But every one of these is **conditional**: the coverage applies only if you used the built-in guardrails (content filters, the safety systems) and did not deliberately infringe. Indemnity is insurance with a deductible of good practice, not a blanket shield. So you build a **compliance gate** that mirrors the conditions: is the provenance ledger complete, are the opt-outs excluded, is the memorization filter on, are the licence conflicts resolved, is there human authorship on published outputs? In the worked example four checks pass but one — licence conflicts (from Step 5) — is unmet, so the gate **BLOCKS** the release and the provider indemnity is **void**: skipping the guardrail both stops the ship and forfeits the coverage. And one more truth that closes the loop: a *purely AI-generated* output has no human author, and under US law (the US Copyright Office guidance and Thaler v Perlmutter) a work with no human author has **no copyright** — so you cannot stop others from copying your model’s raw output; only human-authored work is protected. The major providers (OpenAI, Anthropic, Google) all condition their indemnity on their guardrails, which is why the gate exists. The block evaluates the gate, keyless.

> 🛡️ **Analogy**
>
> Provider indemnity is **an insurance policy with conditions in the fine print**. Your home insurance covers a break-in — but read the policy and it covers you only if the alarm was armed and the deadbolt was locked. Leave the door wide open and the claim is denied, not because the burglary was not real, but because you did not meet the conditions of coverage. Copyright indemnity works exactly this way: the provider will defend you, *provided* you kept the content filters on and did not deliberately infringe. The compliance gate is your pre-flight checklist that every condition is met before you ship — because discovering the indemnity was void *after* the claim arrives is the insurance equivalent of finding out you left the door open.

One gate check is unmet (licence conflicts), but your cloud provider offers copyright indemnity. Are you covered?

**📝 `07_compliance_gate.py`** - *runnable*

In [ ]:
# THE COMPLIANCE GATE + INDEMNITY: providers now offer copyright INDEMNITY (Microsoft Copilot Copyright Commitment, Google, OpenAI
# Copyright Shield) - but ONLY if you use the guardrails. Gate the deploy on them, and know that AI-only outputs are not copyrightable.
CHECKS = {                                          # each must hold for the deploy to ship and the indemnity to apply
    "provenance ledger complete": True,
    "TDM opt-outs excluded": True,
    "output memorization filter on": True,
    "licence conflicts resolved": False,            # a GPL dependency is unresolved (from step 5)
    "human authorship on published outputs": True}  # US Copyright Office: pure-AI output has no copyright
def gate(checks):
    return [name for name, ok in checks.items() if not ok]
failures = gate(CHECKS)
print("COPYRIGHT COMPLIANCE GATE (model release):")
for name, ok in CHECKS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS - ship; provider indemnity applies" if not failures else "BLOCK - {} unmet, indemnity VOID:".format(len(failures))))
for f in failures:
    print("   - " + f)
print("Indemnity is conditional: skip the guardrails and you lose the coverage AND take the liability. And remember - a pure-AI")
print("output is not copyrightable (US Copyright Office), so you cannot stop others from copying it; only human-authored work is.")

# Output:
# COPYRIGHT COMPLIANCE GATE (model release):
#   [x] provenance ledger complete
#   [x] TDM opt-outs excluded
#   [x] output memorization filter on
#   [ ] licence conflicts resolved
#   [x] human authorship on published outputs
#
# gate: BLOCK - 1 unmet, indemnity VOID:
#    - licence conflicts resolved
# Indemnity is conditional: skip the guardrails and you lose the coverage AND take the liability. And remember - a pure-AI
# output is not copyrightable (US Copyright Office), so you cannot stop others from copying it; only human-authored work is.

- The gate mirrors the indemnity conditions: provenance complete, opt-outs excluded, memorization filter on, licences resolved, human authorship.
- Four checks pass but one — licence conflicts (Step 5) — is unmet, so the gate **BLOCKS** the release.
- Because indemnity is conditional on the guardrails, the unmet check makes the provider indemnity **void**.
- And a pure-AI output has no human author, so it has **no copyright** (US Copyright Office) — you cannot stop others copying it.

#### 💡 What just happened

⚡ What just happened? The gate ties the six checks together: one unmet condition (licence conflicts) BLOCKS the release and voids the conditional provider indemnity - and a pure-AI output has no copyright at all. Tradeoff: the gate can block a launch on a paperwork failure, which is the point - it fails safe. That is the whole toolkit: quantify exposure, filter memorization, weigh fair use, measure similarity, check licences, keep provenance, and gate on indemnity.

> 📦 **Info**
>
> 🚫 Two things the gate does not do for youThe compliance gate is necessary, not sufficient, and it is not legal advice. First, **indemnity is not a strategy**: relying on the provider to defend you is a backstop, not a substitute for the controls — the coverage is capped, conditional, and does not cover reputational harm or an injunction that pulls your product. Second, **a green gate is not a legal opinion**: every threshold in this lesson (the similarity line, the factor scores, the memorization length) is an engineering heuristic that surfaces risk for a human to judge, not a verdict a court would issue. The correct posture is to build these checks into your release *and* keep counsel in the loop for the genuinely contested calls. The engineer’s job is to make the risks visible and the easy failures impossible — not to render the legal judgement.

- A five-item release gate with one unmet check
- The gate BLOCKS and the provider indemnity is voided

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — copyright as a release gate
> Copyright in GenAI is not one question but a pipeline of checks from training to release. **Quantify the exposure** — how much of your corpus is copyrighted and unlicensed (Step 1). **Filter memorization** — block outputs that reproduce a source verbatim (Step 2). **Weigh fair use** — score the four factors as a balance, with market harm heaviest (Step 3). **Measure substantial similarity** — catch the non-verbatim paraphrase that keeps the protected expression (Step 4). **Check licence compatibility** — make sure copyleft and non-commercial terms do not sink the stack (Step 5). **Keep provenance** — a per-document ledger so you can honour opt-outs and produce a training-data summary (Step 6). And **gate the release** on all of it, because the conditions that protect you are the same ones that keep your indemnity valid (Step 7). Ask two questions before you ship: have I made the copyright risks *visible*, and have I made the *easy* failures — verbatim output, an opted-out source, a licence conflict — impossible?

**📦 **The copyright risk-control map****

| Risk | The check | The control |
|---|---|---|
| Training on copyrighted data | Corpus exposure percentage | Prefer licensed / public-domain data; size the risk |
| Verbatim memorization | Longest shared word-run vs threshold | Output-side filter blocks the passage |
| Substantial similarity | Distinctive-content overlap vs de-minimis | Flag and review; keep expression, not just words, distinct |
| Licence incompatibility | Copyleft / non-commercial conflict scan | Resolve in CI before shipping |
| Opt-out / provenance | Per-document ledger, TDM opt-out flag | Exclude opted-out sources; Art. 53 summary |
| No indemnity coverage | Compliance gate on the conditions | Meet the guardrails; gate the release |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now treat copyright as a set of engineering checks rather than a launch-day surprise. This builds directly on explainability from Lesson 15.2 — both are axes of accountability, one for the model’s reasoning and one for its data and outputs. What the model learned about the specific *people* in its training data — names, faces, secrets — is a privacy question we take up in Lesson 15.4. Proving how a piece of media was made, and the deepfake and watermarking problem, is Lesson 15.5 (where C2PA provenance signing returns). The regime that turns these obligations into law — the EU AI Act risk tiers and the training-data summary — is the regulatory landscape in Lesson 15.6, and the governance program that owns this compliance gate and assigns who runs it closes the module in Lesson 15.7. The reference tooling is the [SPDX license list](https://github.com/spdx/license-list-data) and the [C2PA specification](https://github.com/c2pa-org/c2pa-specifications); the governing text is the [EU AI Act Article 53](https://artificialintelligenceact.eu/article/53/). This is engineering education, not legal advice — keep counsel in the loop for the contested calls.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Copyright & Intellectual Property— Copyright as Engineering Checks**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-15_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-15.3.html` - regenerate this notebook whenever the source HTML is updated.*
